# ShopSense — CLIP Embedding Test
All backend ML testing happens in Colab. This notebook verifies that CLIP (`openai/clip-vit-base-patch32`) loads and produces 512-dim L2-normalized image embeddings — the same function the FastAPI `/embed` endpoint uses (`app/embeddings.py`).

In [ ]:
!pip install -q transformers torch pillow

In [ ]:
import time
from transformers import CLIPProcessor, CLIPModel

t0 = time.time()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
model.eval()
print(f"CLIP loaded in {time.time()-t0:.1f}s")

## 1. Upload a product image
Apni koi product screenshot/photo upload karein (misal: kala joota, handbag).

In [ ]:
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()
assert uploaded, "Koi image upload nahi hui — dobara try karein."
name = list(uploaded.keys())[0]
img = Image.open(io.BytesIO(uploaded[name])).convert("RGB")
print(f"Loaded: {name}  size={img.size}")
img.thumbnail((512, 512))
img

## 2. Embedding nikalein

In [ ]:
import torch, numpy as np

def embed(image):
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
    # newer transformers return BaseModelOutputWithPooling, not a raw tensor
    feats = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats[0].detach().cpu().numpy()

t0 = time.time()
vec = embed(img)
dt = time.time() - t0
print(f"dim        : {vec.shape[0]}")
print(f"time       : {dt*1000:.0f} ms")
print(f"L2 norm    : {np.linalg.norm(vec):.4f}  (1.0 hona chahiye — normalized)")
print(f"first 5    : {vec[:5].round(4)}")

## 3. Sanity check — cosine similarity
Ussi image ko dobara embed karke similarity 1.0 aani chahiye.

In [ ]:
vec2 = embed(img)
sim = float(np.dot(vec, vec2))
print(f"cosine similarity (same image): {sim:.4f}")
assert abs(sim - 1.0) < 1e-4, "Similarity 1.0 nahi aayi — kuch garbar hai!"
print("PASS: embedding deterministic hai.")

## 4. (Optional) Do mukhtalif images compare karein
Doosri image upload karke dekhein ke milti-julti cheezon ki similarity zyada aati hai ya nahi.

In [ ]:
# Optional cell — chalane ke liye uncomment karein
# uploaded2 = files.upload()
# name2 = list(uploaded2.keys())[0]
# img2 = Image.open(io.BytesIO(uploaded2[name2])).convert("RGB")
# sim2 = float(np.dot(vec, embed(img2)))
# print(f"similarity image1 vs image2: {sim2:.4f}")